In [ ]:
from transformers import (
    TapasTokenizer, TapasForQuestionAnswering,
)
from datasets import load_dataset
import torch
import pandas as pd
from tqdm import tqdm

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/wikitablequestions", trust_remote_code=True)
test_data = dataset["test"]

print("Number of test samples:", len(test_data))
print("Example:", test_data[0])


Number of test samples: 4344
Example: {'id': 'nu-0', 'question': 'which country had the most cyclists finish within the top 10?', 'answers': ['Italy'], 'table': {'header': ['Rank', 'Cyclist', 'Team', 'Time', 'UCI ProTour\\nPoints'], 'rows': [['1', 'Alejandro Valverde\xa0(ESP)', "Caisse d'Epargne", '5h 29\' 10"', '40'], ['2', 'Alexandr Kolobnev\xa0(RUS)', 'Team CSC Saxo Bank', 's.t.', '30'], ['3', 'Davide Rebellin\xa0(ITA)', 'Gerolsteiner', 's.t.', '25'], ['4', 'Paolo Bettini\xa0(ITA)', 'Quick Step', 's.t.', '20'], ['5', 'Franco Pellizotti\xa0(ITA)', 'Liquigas', 's.t.', '15'], ['6', 'Denis Menchov\xa0(RUS)', 'Rabobank', 's.t.', '11'], ['7', 'Samuel Sánchez\xa0(ESP)', 'Euskaltel-Euskadi', 's.t.', '7'], ['8', 'Stéphane Goubert\xa0(FRA)', 'Ag2r-La Mondiale', '+ 2"', '5'], ['9', 'Haimar Zubeldia\xa0(ESP)', 'Euskaltel-Euskadi', '+ 2"', '3'], ['10', 'David Moncoutié\xa0(FRA)', 'Cofidis', '+ 2"', '1']], 'name': 'csv/203-csv/733.tsv'}}


In [4]:
print("\n=== TAPAS ===")
tapas_tokenizer = TapasTokenizer.from_pretrained("google/tapas-base-finetuned-wtq")
tapas_model = TapasForQuestionAnswering.from_pretrained("google/tapas-base-finetuned-wtq")


=== TAPAS ===


The code below is a basic test with like 30% accuracy but the paper says it can reach 48%. And we used the wtq finetuned for this test so the problem is most likely coming from us (except if the base model doesnt perform as well i need to double check that). Anyways maybe a better config could do the trick, see here: https://huggingface.co/docs/transformers/model_doc/tapas

In [19]:
def test_model(model, tokenizer, data):
    correct = 0
    total = len(data)
    total_skipped = 0
    print("Evaluating on", total, "samples...")


    for ex in tqdm(data, desc="Evaluating"):
        headers = ex["table"]["header"]
        rows = ex["table"]["rows"]

        # handle irregular row lengths
        max_len = len(headers)
        clean_rows = [r + [""] * (max_len - len(r)) for r in rows]
        table = pd.DataFrame(clean_rows, columns=headers)
        question = ex["question"]

        # Tokenize 
        inputs = tokenizer(
            table=table,
            queries=[question],
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        # Run model
        try:
            with torch.no_grad():
                outputs = model(**inputs)
        except IndexError:
            print("Skipping table due to token/embedding overflow")
            total_skipped += 1
            continue
        logits = outputs.logits.cpu()

        # Convert logits → predictions
        predictions = tokenizer.convert_logits_to_predictions(inputs, logits)
        if len(predictions) == 2:
            predicted_answer_coordinates, _ = predictions
        else:
            predicted_answer_coordinates = predictions[0]

        if predicted_answer_coordinates[0]:
            answers = [table.iat[row, col] for row, col in predicted_answer_coordinates[0]]
        else:
            answers = []

        if set(answers) == set(ex["answers"]):
            correct += 1

    return correct, total, total_skipped

In [17]:
dev_test = dataset["validation"] # this is what they use in the paper for the accuracy

print("Number of test samples:", len(dev_test))

Number of test samples: 2831


In [20]:
correct, total, total_skipped = test_model(tapas_model, tapas_tokenizer, dev_test)
print(f"Accuracy: {correct / (total - total_skipped) * 100:.2f}%, out of {total - total_skipped} samples (skipped {total_skipped})")

Evaluating on 2831 samples...


Evaluating:  24%|██▍       | 685/2831 [02:42<15:55,  2.25it/s]

Skipping table due to token/embedding overflow


Evaluating:  28%|██▊       | 787/2831 [03:06<24:10,  1.41it/s]

Skipping table due to token/embedding overflow


...

Skipping table due to token/embedding overflow


Evaluating:  84%|████████▍ | 2385/2831 [09:20<05:27,  1.36it/s]

Skipping table due to token/embedding overflow


Evaluating: 100%|██████████| 2831/2831 [11:05<00:00,  4.26it/s]

Accuracy: 29.84%, out of 2825 samples (skipped 6)
